# SAC Irrigation Training - v2.14 (entropy/reward SNR recovery via α=0.01, Colab)

**Algorithm:** SAC (stable_baselines3) with VDN-factorised twin-Q + LayerNorm critic (byte-identical to v2.11)
**Actor:** LeakyReLU(0.01) MLP, trained on NORMALISED global/forecast observations
**Learning rate:** asymmetric - actor LR = 5x critic LR
**Gamma:** 0.99 (unchanged)

## Why v2.12 exists

v2.11 (LayerNorm critic) suppressed the v2.7 deadly-triad cascade but produced a
degenerate, uniform ~5.97 mm/day policy. Direct instrumentation of the trained
v2.11 checkpoints established the proximate cause:

- **Dead-ReLU collapse.** On real observations across a full season, ~0.1 of 128
  first-layer actor units fire (vs ~21/128 in v2.7). Every ReLU outputs ~0, so the
  actor's mu head sees only its bias and emits a constant pre-tanh value -> flat
  ~5.9 mm/day regardless of state.
- **Root cause:** the global scalar + 48-dim forecast blocks were fed RAW
  (rainfall ~11, Kc_ET ~7, radiation ~32) while the per-agent block was normalised
  to [0, 1.5]. The radiation sub-block alone contributed about -3.2 to every
  first-layer pre-activation, dragging all units below zero.
- **Second mechanism (from E4, gamma=0.98):** even with live ReLUs, a stable critic
  gives the actor a weak gradient, so it barely modulates. Addressed with an
  asymmetric (higher) actor LR.

## The v2.12 fix (critic byte-identical to v2.11)
1. Normalise global+forecast block: rainfall/70, Kc_ET/8, radiation/35 (root-cause fix).
2. LeakyReLU(0.01) actor activation (dead-unit insurance).
3. Asymmetric actor LR = 5x critic LR (counters the LayerNorm-bounded critic gradient).

Everything else is identical to v2.11: gamma=0.99, alpha=0.05 fixed, tau=0.005,
batch 256, buffer 250k, 250k steps, standard 1-step ReplayBuffer, MAX_GRAD_NORM=1.0.


**v2.13 update:** v2.12 cured the literal dead-ReLU but the actor still ran at ~11% capacity because all-positive inputs interacting with downward output pressure (entropy + overshoot penalty) drove weight row-sums strongly negative, putting 117/128 first-layer units in LeakyReLU's negative regime. v2.13 adds ONE surgical change: the actor re-centers its input via `x' = 2*x - 1` inside `_per_agent_features`. Empirically verified to lift alive-unit count from 11% to >50% on real obs. The critic, the env, the reward, and all other hyperparameters are byte-identical to v2.12.


**v2.14 update:** v2.13 cured the actor capacity issue at the architecture level but the actor's pre-tanh mu range stayed in [-0.20, +0.08] (timid), producing actions only in [5.0, 6.5] mm/day. Tracing v2.7's 10 saved checkpoints revealed v2.7 was timid pre-cascade and only became responsive (mu_std 0.31, range [0.3, 9.7] mm/day, peak yield 3716) when the deadly-triad cascade pushed per-agent |Q| above 10, raising the SNR Q/(α·H) above ~250. v2.13's stable Q ≈ 3 gives SNR ≈ 80 at α=0.05. v2.14 lowers α to 0.01 to raise SNR back to the v2.7-200k regime without re-introducing the cascade. Architecturally byte-identical to v2.13.


In [ ]:
# Cell 1: Mount Google Drive, clone repo, install deps.
import subprocess, sys, os

from google.colab import drive
drive.mount('/content/drive', force_remount=False)
DRIVE_ROOT = '/content/drive/MyDrive/thesis_results'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print(f'Drive mounted. Results -> {DRIVE_ROOT}')

if os.path.exists('/content/thesis'):
    subprocess.run(['rm', '-rf', '/content/thesis'], check=True)
subprocess.run(
    ['git', 'clone', 'https://github.com/taratorbati/thesis.git', '/content/thesis'],
    check=True)

os.chdir('/content/thesis')
sys.path.insert(0, '/content/thesis')

# Install SB3 - sb3-contrib NOT required for v2.12 (no TQC).
subprocess.run(
    ['pip', 'install', '--quiet',
     'stable-baselines3==2.6.0', 'gymnasium', 'wandb', 'pytest'],
    check=True)

import torch
print(f'PyTorch:        {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:            {torch.cuda.get_device_name(0)}')


In [ ]:
# Cell 2: WandB secret + GPU check.
import os
try:
    from google.colab import userdata
    os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
    print('OK  WANDB_API_KEY loaded from Colab Secrets.')
except Exception as e:
    print(f'NOTE: Could not load WANDB_API_KEY ({type(e).__name__}).')
    print('     Training continues without WandB - add the key to Colab Secrets to enable it.')

import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(r.stdout if r.returncode == 0 else 'nvidia-smi failed - no GPU allocated')


In [ ]:
# Cell 3: Pre-training validation.
#
# Runs smoke tests, factorized-critic tests (now including v2.12 LeakyReLU actor,
# obs-norm marker, and normalised-globals guards), and a 1000-step pilot to catch
# import/wiring bugs.  Abort if anything fails.
import subprocess, sys

print('Smoke tests...')
r = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests/test_rl_smoke.py', '-v', '--tb=short'],
    capture_output=False)
assert r.returncode == 0, 'SMOKE TESTS FAILED'

print('\nFactorized-critic tests (v2.7 + v2.11 + v2.12 architectures)...')
r = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests/test_factorized_critic.py', '-v', '--tb=short'],
    capture_output=False)
assert r.returncode == 0, 'FACTORIZED CRITIC TESTS FAILED'

print('\n1000-step pilot training (wiring check, ~1-2 min)...')
from src.rl.train_v214 import train_sac_v214
_ = train_sac_v214(
    seed=999,
    output_dir='/content/pilot',
    wandb_project=None,
    total_timesteps=1000,
)
print('\nOK  Pre-flight passed. Proceed to Cell 4.')


In [ ]:
# Cell 4: Full 250k training (SAC v2.14 (entropy/reward SNR recovery via α=0.01), gamma=0.99).
# ~30-55 min on A100, ~2-2.5 h on T4.
#
# Start with SEED=0 (paired with v2.7 / v2.11 seed 0). Expand to seeds 1, 2 only
# after seed-0 meets the acceptance criterion.

SEED = 0       # CHANGE per session

from src.rl.train_v214 import train_sac_v214

model = train_sac_v214(
    seed=SEED,
    output_dir='/content/thesis/results/rl',
    wandb_project='sac-irrigation-thesis',
    total_timesteps=250_000,
    gamma=0.99,
    actor_lr_mult=5.0,   # actor LR = 5x critic LR
)
print('Training complete.')


In [ ]:
# Cell 5: Copy results to Google Drive (excluding replay buffer).
import shutil, os, datetime

src = f'/content/thesis/results/rl/sac_v214_seed{SEED}'
timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
dst = f'{DRIVE_ROOT}/sac_v214_seed{SEED}_{timestamp}'

shutil.copytree(src, dst, ignore=shutil.ignore_patterns('replay_buffer_latest.pkl'))
print(f'Saved to: {dst}')
print()
for root, _, files in os.walk(dst):
    for f in files:
        p = os.path.join(root, f); size = os.path.getsize(p)
        print(f'  {os.path.relpath(p, dst)}  ({size/1024:.1f} KB)')


In [ ]:
# Cell 6: Post-training 9-cell evaluation (SAC eval path).
#
# v2.12 produces a SAC checkpoint. The runner auto-detects it via the
# 'actor.obs_norm_marker' buffer (+ the 1-D 'critic.qf0.1.weight' LayerNorm key)
# and dispatches to V214CTDESACPolicy, AND applies the SAME global/forecast
# normalisation at eval time (normalize_globals=True). This train/eval match is
# essential - it was verified to be byte-exact.
import subprocess, sys

model_path = f'/content/thesis/results/rl/sac_v214_seed{SEED}/best_model/best_model.zip'

print('Evaluating on 9-cell grid (perfect forecast)...')
r = subprocess.run([
    sys.executable, '-m', 'scripts.experiments.exp_rl',
    '--mode',     'eval',
    '--model',    model_path,
    '--scenario', 'all',
    '--budget',   'all',
    '--forecast', 'perfect',
], capture_output=False)
assert r.returncode == 0, 'PERFECT-FORECAST EVAL FAILED'

print('\nEvaluating on 9-cell grid (noisy forecast, seed=42)...')
r = subprocess.run([
    sys.executable, '-m', 'scripts.experiments.exp_rl',
    '--mode',       'eval',
    '--model',      model_path,
    '--scenario',   'all',
    '--budget',     'all',
    '--forecast',   'noisy',
    '--noise-seed', '42',
], capture_output=False)
if r.returncode != 0:
    print('Noisy-forecast eval failed; perfect-forecast only.')


In [ ]:
# Cell 7: 'Did the actor wake up?' diagnostic (the v2.12 acceptance check).
#
# This is the decisive plot. It loads each saved checkpoint, runs a full season of
# REAL observations through the actor, and measures:
#   (a) first-layer live-unit count  (v2.11 was ~0/128; target > 30/128)
#   (b) pre-tanh mu temporal std across the season (v2.11 ~0.001; target >> 0.05)
#   (c) mu.weight std growth          (v2.11 shrank to 0.027; target grows past 0.10)
import numpy as np, torch, glob, os, re
import matplotlib.pyplot as plt
import torch.nn as nn
from src.rl.gym_env import IrrigationEnv

ckpt_dir = f'/content/thesis/results/rl/sac_v214_seed{SEED}/checkpoints'
paths = sorted(glob.glob(os.path.join(ckpt_dir, f'sac_v214_seed{SEED}_*_steps.zip')),
               key=lambda p: int(re.search(r'_(\d+)_steps', p).group(1)))

def load_actor(zip_path):
    import zipfile, io
    with zipfile.ZipFile(zip_path) as zf:
        sd = torch.load(io.BytesIO(zf.read('policy.pth')), map_location='cpu', weights_only=False)
    a = nn.ModuleDict({
        'l0': nn.Linear(65,128), 'l2': nn.Linear(128,128), 'mu': nn.Linear(128,1)})
    a['l0'].weight.data = sd['actor.latent_pi.0.weight']; a['l0'].bias.data = sd['actor.latent_pi.0.bias']
    a['l2'].weight.data = sd['actor.latent_pi.2.weight']; a['l2'].bias.data = sd['actor.latent_pi.2.bias']
    a['mu'].weight.data = sd['actor.mu.weight'];          a['mu'].bias.data = sd['actor.mu.bias']
    return a, sd

def o2p(obs, N=130, F=8):
    per = obs[:F*N].reshape(N,F); g = obs[F*N:]
    return np.concatenate([per, np.broadcast_to(g,(N,g.shape[0]))], axis=1).astype(np.float32)

env = IrrigationEnv(randomize=False, curriculum_warmup_steps=0,
                    use_overshoot_feature=False, normalize_globals=True)
steps, alive_l, mutstd_l, muwstd_l = [], [], [], []
for p in paths:
    actor, sd = load_actor(p)
    obs,_ = env.reset(); alive=[]; mus=[]
    for d in range(93):
        pa = torch.from_numpy(o2p(obs))
        h1 = torch.relu(actor['l0'](pa))                       # LeakyReLU>0 == ReLU>0 for liveness
        mu = actor['mu'](torch.nn.functional.leaky_relu(actor['l2'](torch.nn.functional.leaky_relu(actor['l0'](pa),0.01)),0.01))
        alive.append((h1>0).any(dim=0).sum().item())
        mus.append(mu.mean().item())
        a = (np.tanh(mu.detach().numpy().flatten())*0.5+0.5)
        obs,_,term,trunc,_ = env.step(a)
        if term or trunc: obs,_ = env.reset()
    step = int(re.search(r'_(\d+)_steps', p).group(1))
    steps.append(step); alive_l.append(np.mean(alive))
    mutstd_l.append(np.std(mus)); muwstd_l.append(sd['actor.mu.weight'].std().item())
    print(f'step {step:>7}: live~{np.mean(alive):5.1f}/128  mu_temporal_std={np.std(mus):.4f}  mu.w_std={sd["actor.mu.weight"].std().item():.4f}')

fig, ax = plt.subplots(1, 3, figsize=(15,4))
ax[0].plot(steps, alive_l, '-o'); ax[0].axhline(30, color='r', ls=':'); ax[0].set_title('live first-layer units / 128'); ax[0].set_xlabel('step'); ax[0].grid(alpha=.3)
ax[1].plot(steps, mutstd_l, '-o'); ax[1].axhline(0.05, color='r', ls=':'); ax[1].set_title('pre-tanh mu temporal std (state response)'); ax[1].set_xlabel('step'); ax[1].grid(alpha=.3)
ax[2].plot(steps, muwstd_l, '-o'); ax[2].axhline(0.10, color='r', ls=':'); ax[2].set_title('mu.weight std (actor learning)'); ax[2].set_xlabel('step'); ax[2].grid(alpha=.3)
plt.tight_layout(); plt.show()
print('\nPASS if: live units rise above ~30, mu temporal std rises well past 0.05,')
print('and mu.weight std grows past ~0.10. (v2.11 stayed flat at ~0, ~0.001, ~0.027.)')


In [ ]:
# Cell 8: Resume from Drive checkpoint (if session was interrupted).
# Uncomment and fill in CHECKPOINT_STEP / CHECKPOINT_DRIVE_PATH.

# SEED = 0
# CHECKPOINT_STEP = 100_000
# CHECKPOINT_DRIVE_PATH = f'{DRIVE_ROOT}/sac_v214_seed{SEED}_YYYYMMDD_HHMMSS'
#
# import shutil, os
# local_dir = f'/content/thesis/results/rl/sac_v214_seed{SEED}'
# os.makedirs(local_dir, exist_ok=True)
# shutil.copytree(CHECKPOINT_DRIVE_PATH, local_dir, dirs_exist_ok=True)
#
# from src.rl.train_v212 import AsymmetricLRSAC
# from src.rl.networks import V214CTDESACPolicy
# ckpt = f'{local_dir}/checkpoints/sac_v214_seed{SEED}_{CHECKPOINT_STEP}_steps.zip'
# model = AsymmetricLRSAC.load(ckpt, custom_objects={'policy_class': V214CTDESACPolicy})
# # Continue: model.learn(total_timesteps=..., reset_num_timesteps=False)
